# meteo-cache: load weather into the cache and read it

This notebook presents the API of `meteo_cache` on a small example: hourly weather for a few locations over an
interval of months, fetched from the Open-Meteo historical archive into an on-disk cache.

1. Set the parameters below and run all cells.
2. The notebook converts the locations to weather cells, reports the days the cache lacks and the API calls
   needed, and then loads them within a time budget (`RUN_MINUTES`).
3. Every completed request is stored at once, so a run that stops (budget, API limit, interruption) loses
   nothing. Run the notebook again to continue; a complete cache sends no request.
4. The cached weather is then read as one table, in UTC and in local time.

The interval is in UTC: `FIRST_MONTH` starts on its first day at 00:00 UTC and `LAST_MONTH` ends on its last
day at 23:00 UTC. Days less than 7 days old are not requested, as the archive is not yet complete for them.

Weather data by [Open-Meteo.com](https://open-meteo.com/) (CC BY 4.0).

## Parameters

- `LOCATIONS`: `{name: (latitude, longitude)}`; each location is mapped to its weather cell (0.1 degree).
- `FIRST_MONTH`, `LAST_MONTH`: the interval, as `"YYYY-MM"`; both months are included.
- `VARIABLES`: the API variables to load; `DEFAULT_VARIABLES` are temperature, relative humidity,
  precipitation and wind speed.
- `UTC_OFFSET_HOURS`: local time minus UTC, used only by the local-time section (1 for CET).
- `RUN_MINUTES`, `CALLS_PER_MINUTE`: one run loads for at most `RUN_MINUTES` minutes at this pace.
  30 min × 160 calls/min = 4,800 calls, below the free API's limit of 5,000 calls per hour.
- `DRY_RUN`: `True` reports the missing days and the plan without sending any request.

In [ ]:
from meteo_cache import DEFAULT_VARIABLES

LOCATIONS = {"Warsaw": (52.23, 21.01), "Rome": (41.90, 12.50)}
FIRST_MONTH = "2024-01"
LAST_MONTH = "2024-03"
VARIABLES = DEFAULT_VARIABLES
UTC_OFFSET_HOURS = 1
RUN_MINUTES = 30
CALLS_PER_MINUTE = 160
DRY_RUN = False

In [ ]:
from pathlib import Path

import pandas as pd

from meteo_cache import (
    OpenMeteoArchive, WeatherCache, add_weather_flags, cache_status, cell_key, fetch_weather, missing_days,
    plan_requests, read_weather, run_requests,
)

ROOT = Path("..").resolve()  # repository root, as the notebook runs from notebooks/
CACHE_DIR = ROOT / "data" / "cache" / "weather_cells"  # data/ is excluded from git
SOURCE = OpenMeteoArchive()

## Weather cells

A weather cell is `"lat,lon"` rounded to 0.1 degree; `cell_key` computes it from scalars or from two pandas
Series. Locations closer than about 10 km may share a cell, and then share its weather.

In [ ]:
names = {cell_key(lat, lon): name for name, (lat, lon) in LOCATIONS.items()}
cells = list(names)
pd.DataFrame({"location": names.values(), "cell": cells})

## The cache

`WeatherCache.open` only reads; the directory is created by the first request that stores data. `check` stops
here if the cache was written by another source or with other column names or units.

In [ ]:
cache = WeatherCache.open(CACHE_DIR)
cache.check(SOURCE, VARIABLES)

print(f"{len(cache.cells())} cells in {CACHE_DIR.relative_to(ROOT)}")
cache.status()

## Missing days in the interval

`periods` is `{cell: (start, end)}` in UTC, both bounds included; the order of the cells is the order of the
requests. `missing_days` counts, per cell, the wanted days and those that lack at least one variable.

In [ ]:
start = pd.Period(FIRST_MONTH, "M").start_time
end = pd.Period(LAST_MONTH, "M").end_time.floor("h")
periods = {cell: (start, end) for cell in cells}

missing = missing_days(cache, periods, VARIABLES, source=SOURCE)
print(f"interval {start:%Y-%m-%d %H:%M} to {end:%Y-%m-%d %H:%M} UTC")
print(f"missing cell-days: {missing['missing_days'].sum():,} of {missing['wanted_days'].sum():,}")
missing

## Plan

`plan_requests` lists the requests that fill the gaps and sends nothing: one `Request` per run of consecutive
missing days, with its cost in API calls (`(variables / 10) × (days / 14)`, each factor at least 1).

In [ ]:
plan = plan_requests(cache, periods, VARIABLES, source=SOURCE, calls_per_minute=CALLS_PER_MINUTE)
calls = sum(r.cost for r in plan)
print(f"{len(plan)} requests for {len({r.cell for r in plan})} cells, {calls:,.1f} API calls, "
      f"approximately {calls / CALLS_PER_MINUTE:.1f} min")
pd.DataFrame(plan)

## Load

`run_requests` sends the plan in order and stores each answer as soon as it arrives. It stops when
`RUN_MINUTES` have passed or when the API reports its hourly or daily limit; `RunResult.stopped` gives the
reason, and everything loaded until then is kept in the cache.

In [ ]:
if DRY_RUN or not plan:
    print("nothing sent:", "DRY_RUN is set" if plan else "the cache is complete for the interval")
else:
    run = run_requests(plan, cache, source=SOURCE, calls_per_minute=CALLS_PER_MINUTE, max_minutes=RUN_MINUTES)
    print(f"{run.done} of {len(plan)} requests completed, {run.spent:,.1f} API calls; "
          f"stopped: {run.stopped or 'no, every request completed'}")

## Result

`cache_status` summarizes a cache directory without opening it for writing: one row per cell and variable,
with the number of days and the first and last UTC day.

In [ ]:
after = missing_days(cache, periods, VARIABLES, source=SOURCE)
print(f"complete cells: {(after['missing_days'] == 0).sum()} of {len(cells)}")
cache_status(CACHE_DIR)

## Read the weather (UTC)

`read_weather` reads the cached hours of the periods as one table and fetches nothing. Hours the cache lacks
are absent rows, not rows of NaN.

In [ ]:
weather = read_weather(cache, periods, VARIABLES)
weather.insert(1, "location", weather["cell"].map(names))
print(f"{len(weather):,} hours")
weather.head()

In [ ]:
monthly = weather.groupby(["location", weather["t_utc"].dt.to_period("M")]).agg(
    air_temperature=("air_temperature", "mean"),
    precipitation=("precipitation", "sum"),
    wind_speed=("wind_speed", "mean"),
)
monthly.round(1)  # degF, inch, mph

## Local time in one call

`fetch_weather` runs the four steps (plan, load, read) and converts the timestamps to local time with a fixed
`utc_offset_hours`; `start` and `end` are then local times. Only the days the cache lacks are requested, so
after the load above it sends nothing. `cache_path=None` keeps the cache in memory.

In [ ]:
if DRY_RUN:
    print("skipped: DRY_RUN is set, and fetch_weather would send the missing requests")
else:
    local_start = start + pd.Timedelta(hours=UTC_OFFSET_HOURS)
    local_end = end + pd.Timedelta(hours=UTC_OFFSET_HOURS)
    local = fetch_weather(cells, local_start, local_end, VARIABLES, cache_path=CACHE_DIR,
                          utc_offset_hours=UTC_OFFSET_HOURS, calls_per_minute=CALLS_PER_MINUTE,
                          max_minutes=RUN_MINUTES, source=SOURCE)
    display(local.head())

## Weather flags

`add_weather_flags` adds three boolean columns: `freezing_precip`, `snowing` and `wet_below_freezing`. The
first two need the variable `weather_code`, which is not among `DEFAULT_VARIABLES`; without it they are
`False` throughout, with a warning.

In [ ]:
flags = add_weather_flags(weather)
flags.groupby("location")[["freezing_precip", "snowing", "wet_below_freezing"]].sum()  # hours